# Check convergence 

In [1]:
import pandas as pd
import numpy as np

from os import listdir
import os
import itertools

from reframed.community.model import Community
from reframed import Environment, set_default_solver
from steadiercom.cli import load_communities,load_media_db

import sys
sys.path.append("../functions/")
import general
from convergence import SteadierSampleConvergence
sys.path.append("../scripts_python")
import convergence_test_script
from IPython.display import HTML, display
from IPython.display import display, HTML

In [2]:
%load_ext autoreload
%autoreload 2

### Gut microbiome community

Concentration $X_0$ (right after serial transfer) and $X_1$ (right before next transfer) in $t_0$ and $t_1$

$$
X_1 = X_0 *e^{\mu t_1} , X_0 = X_0
$$

Assuming that the cells grow back to the original concentration (after being diluted 1:20 => the cells increase in density by 20 times)

$$
X_1/X_0 = 20 = e^{\mu t} 
$$

The above equation turns into:
$$
ln(X_1/X_0) = \mu t
$$

Which means the avearage growth rate becomes
$$
\mu_{avg} = \frac{ln(20)}{t}
$$

**Methods (From Venturelli et al. 2018): 1:20 dilution every 24 h**

In [3]:
np.log(20)/24

np.float64(0.12482217806474961)

### Select models for convergence test

In [4]:
models_gut = general.load_models_venturelli2018()

INFO     | Loading models from Venturelli et al (2018)...


**Find largest models in terms of metabolites**

In [5]:
size={}
for name, model in models_gut.items():
    size[name] = len(model.reactions)
pd.Series(size).sort_values(ascending=False).head(5)

Bacteroides_ovatus_ATCC_8483             2380
Bacteroides_uniformis_ATCC_8492          2154
Bacteroides_thetaiotaomicron_VPI_5482    2065
Bacteroides_vulgatus_ATCC_8482           1894
Faecalibacterium_prausnitzii_A2_165      1599
dtype: int64

**Find members with largest external metabolite overlap**

In [6]:
overlap={}
for name1, name2 in itertools.combinations(models_gut.keys(), 2):
    index = tuple(sorted([name1, name2]))
    ext_mets1 = models_gut[name1].get_external_metabolites()
    ext_mets2 = models_gut[name2].get_external_metabolites()
    overlap[index] = len(set(ext_mets1) & set(ext_mets2))
overlap_ser = pd.Series(overlap).sort_values(ascending=False)
members_big_overlap = overlap_ser.index[0]
members_big_overlap

('Bacteroides_ovatus_ATCC_8483', 'Bacteroides_thetaiotaomicron_VPI_5482')

**Make dataframe**

In [7]:
communities_exp = pd.read_csv("../data/Venturelli2018/generated/community_df_pairwise_gut.tsv",sep="\t",header=None)
community_id = communities_exp.loc[communities_exp[1].isin(members_big_overlap), 0]\
                                                                .value_counts()\
                                                                .idxmax()

communities_exp[communities_exp[0]==community_id].to_csv("../data/Venturelli2018/generated/communities_convergence.tsv",
                                                                                                    sep="\t", 
                                                                                                    index=False,
                                                                                                    header=False)

communities_exp.loc[communities_exp[0]==community_id,
                    communities_exp.columns[:-1]].to_csv("../data/Venturelli2018/generated/communities_convergence_no_abundance.tsv",
                                                                                                    sep="\t", 
                                                                                                    index=False,
                                                                                                    header=False)


### Convergence for pairwise study with pairwise gut bacteria - without fixed abundance

In [8]:
mediadb = "../data/Venturelli2018/generated/media_db.tsv"#path to mediadb
medium = "LB_extend"
growth = np.log(20)/24# average growth rate

models = ["../models/Venturelli2018/species_used_noO2/"+name for name in listdir("../models/Venturelli2018/species_used_noO2/")]
communities = "../data/Venturelli2018/generated/communities_convergence.tsv"
output_dir = "../output/Venturelli2018/convergence_test_pairwise/"
comm_id=community_id

In [9]:
dfs_tot, dfs_tot_abs = convergence_test_script.load_data_and_preprocess(output_dir)


INFO     | Loading data from ../output/Venturelli2018/convergence_test_pairwise/...


In [10]:

results = convergence_test_script.process_data(dfs_tot_abs)



INFO     | Processing data (finding standard deviation and coefficient of variance)...


250
500
1000
2000
4000
8000
500
1000
1500
2000
2500
3000
3500
4000
4500
5000


In [11]:
convergence_test_script.make_report(results,output_dir,ecoli_or_gut="Venturelli2018")

INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/pandas/core/algorithms.py:1743: RuntimeWarning: divide by zero encountered in log10
  return lib.map_infer(values

In [12]:

html_path = os.path.join("html_reports/Venturelli2018/", output_dir.split("/")[-2], "report.html")
display(HTML(filename=html_path))

### Convergence for pairwise study with pairwise gut bacteria - with fixed abundance

In [13]:
mediadb = "../data/Venturelli2018/generated/media_db.tsv"#path to mediadb
medium = "LB_extend"
growth = np.log(20)/24# average growth rate

models = ["../models/Venturelli2018/species_used_noO2/"+name for name in listdir("../models/Venturelli2018/species_used_noO2/")]
communities = "../data/Venturelli2018/generated/communities_convergence_no_abundance.tsv"
output_dir = "../output/Venturelli2018/convergence_test_pairwise_no_abundance/"
comm_id=community_id

In [14]:
dfs_tot, dfs_tot_abs = convergence_test_script.load_data_and_preprocess(output_dir)


INFO     | Loading data from ../output/Venturelli2018/convergence_test_pairwise_no_abundance/...


In [15]:

results = convergence_test_script.process_data(dfs_tot_abs)



INFO     | Processing data (finding standard deviation and coefficient of variance)...


250
500
1000
2000
4000
8000
500
1000
1500
2000
2500
3000
3500
4000
4500
5000


In [16]:
convergence_test_script.make_report(results,output_dir,ecoli_or_gut="Venturelli2018")

INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/pandas/core/algorithms.py:1743: RuntimeWarning: divide by zero encountered in log10
  return lib.map_infer(values

In [17]:

html_path = os.path.join("html_reports/Venturelli2018/", output_dir.split("/")[-2], "report.html")
display(HTML(filename=html_path))